# Segmentation and data augmentation
This notebook performs segmentation and data augmentation, taking input as a .png file, supposed to be a high-quality scan of the  printed sheet (sheet.pdf) with the black squares filled. 

## Segmentation
By chance, everyone filled the boxes with blue pen, therefore I opted for the following approach for the segmentation: first, the RGB image is converted to grayscale by setting bright and dark pixels to 0, with some tolerance determined by trial and error. This leaves only the (~blue) letters. Then, the original image is again converted to grayscale, this time, by a built-in openCV function. After this, dark pixels are turned to white, and bright pixels to black, according to some threshold value. The boxes are selected by applying a relatively big median filter, which erases all the thin features (such as penstrokes and noise), only keeping the black textboxes. The boxes are selected by a built-in method and the letters in them (from the first image) are cropped to square and resized to 28x28 px. These constitute the data to be augmented.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2


In [4]:
def segment_images(images):
    for i in range(len(images)):
        img = cv2.imread(images[i])
        #written by genAI after specifying the requirements (up to the empty comment)
        lower_black = np.array([0, 0, 0])
        upper_black = np.array([70, 70, 70]) 

        # Define "approximately white" (from light gray up to absolute white)
        lower_white = np.array([220, 220, 220]) 
        upper_white = np.array([255, 255, 255])

        # 3. Create masks for these specific color ranges
        # Pixels within the range become 255 (white) in the mask, others become 0 (black)
        mask_black = cv2.inRange(img, lower_black, upper_black)
        mask_white = cv2.inRange(img, lower_white, upper_white)

        # 4. Combine the masks
        # This creates a single mask where ALL black AND white pixels are marked as 255
        combined_mask = cv2.bitwise_or(mask_black, mask_white)

        # 5. Create the final output image
        # Initialize a completely black, single-channel (grayscale) image of the same size
        gray = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        # We want the "rest" (the pixels NOT in our combined mask) to be white.
        # So, wherever the combined mask is 0 (meaning it wasn't black or white originally),
        # we set the result image to 255 (white).
        gray[combined_mask == 0] = 255
        ###

        #removes "dot" noise from the image
        gray = cv2.medianBlur(gray, 3)

        gray2 = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, gray2 = cv2.threshold(gray2, 160, 255, cv2.THRESH_BINARY_INV)
        gray2 = cv2.medianBlur(gray2, 11)

        contours, hierarchy = cv2.findContours(gray2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sort_contours_by_row(contours, 50)
        drawn = 0
        l = []
        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            if(w*h > 1000):  # Filter out small 
                letter = gray[y+19:y+187, x+19:x+187]
                letter = cv2.resize(letter, (28, 28))
                l.append(x)
                drawn += 1

In [3]:
# Sorting contours so that we know which letter they correspond to.
# Written by genAI

def sort_contours_by_row(contours, y_tolerance=10):
    """
    Sorts contours top-to-bottom, then left-to-right, grouping them by rows.
    
    :param contours: List of contours from cv2.findContours
    :param y_tolerance: Maximum pixel difference in Y to be considered the same row.
                        (A good rule of thumb is half the average height of your contours)
    """
    if not contours:
        return []

    # 1. Get bounding boxes and pair them with contours
    boxes = [cv2.boundingRect(c) for c in contours]
    
    # 2. Sort strictly top-to-bottom first
    sorted_by_y = sorted(zip(contours, boxes), key=lambda b: b[1][1])
    
    rows = []
    current_row = [sorted_by_y[0]]
    
    # 3. Group into rows based on y_tolerance
    for item in sorted_by_y[1:]:
        contour, box = item
        _, y, _, _ = box
        
        # Get Y coordinate of the first element in the current row
        row_y = current_row[0][1][1]
        
        if abs(y - row_y) <= y_tolerance:
            # It's close enough in Y, add to current row
            current_row.append(item)
        else:
            # Row is finished. Sort the completed row strictly by X (left-to-right)
            current_row.sort(key=lambda b: b[1][0])
            rows.extend(current_row)
            
            # Start a new row
            current_row = [item]
            
    # 4. Sort and add the final row
    if current_row:
        current_row.sort(key=lambda b: b[1][0])
        rows.extend(current_row)
        
    # 5. Extract just the sorted contours without the bounding box data
    final_contours = [item[0] for item in rows]
    
    return final_contours